# 00h Anchor vs Synthetic Alignment Check

Purpose:
- compare synthetic BOM-dependent history against the real Excel anchor,
- quantify SKU coverage and value-scale alignment,
- emit a machine-readable quality report for gate discussion.

In [6]:
from pathlib import Path
from datetime import datetime
import json
import numpy as np
import pandas as pd

ROOT = Path('/Users/k.e.oshada/Documents/OptiWMS')
REPORTS_DIR = ROOT / 'Ai miroservices' / 'modeling' / 'outputs' / 'reports'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

ANCHOR_CANDIDATES = [
    Path('/Users/k.e.oshada/Desktop/Mavin sir resources WMS hemas/RM ROP and Pallet requirement - SEP.xlsx'),
    Path('/Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/Forecast model train data optiwms/RM ROP and Pallet requirement  4- SEP.xlsx'),
]

SYNTH_CSV = ROOT / 'ai-services' / 'forecast-service' / 'artifacts' / 'backfill' / 'synthetic_bom_dependent_history.csv'
SYNTH_REPORT = ROOT / 'ai-services' / 'forecast-service' / 'artifacts' / 'backfill' / 'synthetic_bom_dependent_history.report.json'

def resolve_anchor(paths):
    for p in paths:
        if p.exists():
            return p
    raise FileNotFoundError(f'No anchor Excel file found. Checked: {paths}')

ANCHOR_EXCEL = resolve_anchor(ANCHOR_CANDIDATES)
if not SYNTH_CSV.exists():
    raise FileNotFoundError(f'Missing synthetic CSV: {SYNTH_CSV}')

ANCHOR_EXCEL, SYNTH_CSV

(PosixPath('/Users/k.e.oshada/Desktop/Mavin sir resources WMS hemas/RM ROP and Pallet requirement - SEP.xlsx'),
 PosixPath('/Users/k.e.oshada/Documents/OptiWMS/ai-services/forecast-service/artifacts/backfill/synthetic_bom_dependent_history.csv'))

In [7]:
anchor = pd.read_excel(ANCHOR_EXCEL, sheet_name='Active stock')
anchor = anchor.iloc[1:].copy()
anchor.columns = [str(c).strip() for c in anchor.columns]
rename_map = {
    'Material Code': 'sku',
    'Description': 'description',
    'future average': 'future_average',
    'Supply Plan': 'plan_jul',
    'Unnamed: 4': 'plan_aug',
    'Unnamed: 5': 'plan_sep',
    'Unnamed: 6': 'plan_oct',
    'Unnamed: 7': 'plan_nov',
}
anchor = anchor.rename(columns=rename_map)

need = ['sku', 'description', 'future_average', 'plan_jul', 'plan_aug', 'plan_sep', 'plan_oct', 'plan_nov']
need = [c for c in need if c in anchor.columns]
anchor = anchor[need].copy()
anchor['sku'] = pd.to_numeric(anchor['sku'], errors='coerce')
anchor = anchor[anchor['sku'].notna()].copy()
anchor['sku'] = anchor['sku'].astype(int).astype(str)

for c in ['future_average', 'plan_jul', 'plan_aug', 'plan_sep', 'plan_oct', 'plan_nov']:
    if c in anchor.columns:
        anchor[c] = pd.to_numeric(anchor[c], errors='coerce')

plan_cols = [c for c in ['plan_jul', 'plan_aug', 'plan_sep', 'plan_oct', 'plan_nov'] if c in anchor.columns]
if plan_cols:
    anchor['anchor_level'] = anchor[plan_cols].mean(axis=1, skipna=True)
else:
    anchor['anchor_level'] = anchor['future_average']

anchor = anchor[['sku', 'description', 'anchor_level']].drop_duplicates(subset=['sku']).reset_index(drop=True)
anchor.head()

,sku,description,anchor_level
0,100036,CAUSTIC SODA,82464.430255
1,101054,CALCIUM CARBONATE ( GROUND ),64260.360734
2,100098,SORBITOL,63309.930912
3,101293,FLUFF UNTREATED - GOLDEN ISLES G4881,48423.163870
4,100108,TALCUM POWDER,32594.003662


In [8]:
synth = pd.read_csv(SYNTH_CSV)
synth['sku'] = synth['sku'].astype(str).str.strip()
synth['demand_units'] = pd.to_numeric(synth['demand_units'], errors='coerce').fillna(0.0)
synth['demand_date'] = pd.to_datetime(synth['demand_date'], errors='coerce')
synth = synth[synth['demand_date'].notna()].copy()

synth_level = (
    synth.groupby('sku', as_index=False)
    .agg(
        synth_mean=('demand_units', 'mean'),
        synth_min=('demand_units', 'min'),
        synth_max=('demand_units', 'max'),
        months=('demand_date', lambda s: s.dt.to_period('M').nunique()),
    )
)
synth_level.head()

,sku,synth_mean,synth_min,synth_max,months
0,100005,1476.341583,1152.041,2484.133,36
1,100006,2213.201833,1321.199,3317.550,36
2,100010,43.102000,34.907,56.447,36
3,100012,131.300778,99.733,149.599,36
4,100016,1.005778,0.716,1.359,36


In [9]:
aligned = anchor.merge(synth_level, on='sku', how='inner')
aligned['abs_pct_err'] = np.where(
    aligned['anchor_level'].abs() > 1e-9,
    (aligned['synth_mean'] - aligned['anchor_level']).abs() / aligned['anchor_level'].abs(),
    np.nan,
)

anchor_skus = int(anchor['sku'].nunique())
synth_skus = int(synth_level['sku'].nunique())
matched = int(aligned['sku'].nunique())

aligned_nonnull = aligned[aligned['abs_pct_err'].notna()].copy()
if not aligned_nonnull.empty:
    trimmed_threshold = float(aligned_nonnull['abs_pct_err'].quantile(0.95))
    trimmed = aligned_nonnull[aligned_nonnull['abs_pct_err'] <= trimmed_threshold]
    weighted_abs_pct = (
        ((aligned_nonnull['synth_mean'] - aligned_nonnull['anchor_level']).abs().sum()) /
        max(aligned_nonnull['anchor_level'].abs().sum(), 1e-9)
    )
else:
    trimmed = aligned_nonnull
    weighted_abs_pct = np.nan

coverage_ok = matched >= max(1, int(0.5 * anchor_skus))
shape_ok = bool((not np.isnan(weighted_abs_pct)) and weighted_abs_pct <= 0.35)
quality_status = 'ok' if (coverage_ok and shape_ok) else ('warn_scale_misalignment' if coverage_ok else 'warn_low_match_coverage')

summary = {
    'timestamp_utc': datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%SZ'),
    'anchor_excel': str(ANCHOR_EXCEL),
    'synthetic_csv': str(SYNTH_CSV),
    'anchor_sku_count': anchor_skus,
    'synthetic_sku_count': synth_skus,
    'matched_sku_count': matched,
    'anchor_coverage_pct': float((matched / anchor_skus) if anchor_skus else 0.0),
    'synthetic_overlap_pct': float((matched / synth_skus) if synth_skus else 0.0),
    'matched_mape_mean': float(aligned['abs_pct_err'].mean()) if matched else None,
    'matched_mape_trimmed_p95_mean': float(trimmed['abs_pct_err'].mean()) if len(trimmed) else None,
    'matched_mape_p50': float(aligned['abs_pct_err'].median()) if matched else None,
    'matched_mape_p90': float(aligned['abs_pct_err'].quantile(0.9)) if matched else None,
    'weighted_abs_pct_err': float(weighted_abs_pct) if not np.isnan(weighted_abs_pct) else None,
    'match_quality_status': quality_status,
}

if SYNTH_REPORT.exists():
    try:
        rpt = json.loads(SYNTH_REPORT.read_text(encoding='utf-8'))
        summary['generator_checks'] = rpt.get('checks', {})
        summary['generator_dataset_version'] = rpt.get('dataset_version')
    except Exception:
        summary['generator_checks'] = {}

summary

{'timestamp_utc': '2026-04-20T13:41:08Z',
 'anchor_excel': '/Users/k.e.oshada/Desktop/Mavin sir resources WMS hemas/RM ROP and Pallet requirement - SEP.xlsx',
 'synthetic_csv': '/Users/k.e.oshada/Documents/OptiWMS/ai-services/forecast-service/artifacts/backfill/synthetic_bom_dependent_history.csv',
 'anchor_sku_count': 288,
 'synthetic_sku_count': 261,
 'matched_sku_count': 260,
 'anchor_coverage_pct': 0.9027777777777778,
 'synthetic_overlap_pct': 0.9961685823754789,
 'matched_mape_mean': 0.8938726660881658,
 'matched_mape_trimmed_p95_mean': 0.25386323326864474,
 'matched_mape_p50': 0.12760312903986973,
 'matched_mape_p90': 0.965630185587281,
 'weighted_abs_pct_err': 0.1323449316589406,
 'match_quality_status': 'ok',
 'generator_checks': {'anchor_matches': 1,
  'anchor_unmatched': 1,
  'component_types': ['raw_material'],
  'fg_skus_used': 1,
  'months_max': 36,
  'months_min': 36,
  'months_p50': 36.0,
  'non_negative': True,
  'range_clamp_events': 36,
  'rows_daily': 9396,
  'sku_co

In [10]:
stamp = datetime.utcnow().strftime('%Y%m%dT%H%M%SZ')
summary_csv = REPORTS_DIR / 'synthetic_anchor_alignment_summary.csv'
summary_json = REPORTS_DIR / 'synthetic_anchor_alignment_metadata.json'
summary_csv_ts = REPORTS_DIR / f'synthetic_anchor_alignment_summary_{stamp}.csv'
summary_json_ts = REPORTS_DIR / f'synthetic_anchor_alignment_metadata_{stamp}.json'

summary_df = pd.DataFrame([summary])
summary_df.to_csv(summary_csv, index=False)
summary_df.to_csv(summary_csv_ts, index=False)

details = aligned.sort_values('abs_pct_err', ascending=False)
details_csv = REPORTS_DIR / 'synthetic_anchor_alignment_details.csv'
details.to_csv(details_csv, index=False)

payload = {
    'summary': summary,
    'worst_20_matches': details.head(20).to_dict(orient='records'),
}
summary_json.write_text(json.dumps(payload, indent=2), encoding='utf-8')
summary_json_ts.write_text(json.dumps(payload, indent=2), encoding='utf-8')

print('WROTE', summary_csv)
print('WROTE', summary_json)
print('WROTE', details_csv)
print('WROTE', summary_csv_ts)
print('WROTE', summary_json_ts)